In [5]:
from pathlib import Path
from datasets import load_dataset
import pandas as pd
import json
from collections import defaultdict

In [6]:
def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in [start, *start.parents]:
        if (path / ".git").exists():
            return path
    raise FileNotFoundError(f"No .git folder found above {start}")

ROOT = find_repo_root()
DATA_DIR = ROOT / "data"
CORPUS_PATH = DATA_DIR / "corpus" / "pooled_corpus.parquet"
SPLITS_DIR = ROOT / "splits"

SEED = 27

# The dataset will be 80% Brigde and 20% Comparison Questions
SLICE_COUNTS = {
    "pilot": {"bridge": 160,    "comparison": 40},
    "tune": {"bridge": 800,    "comparison": 200},
    "probe_train": {"bridge": 8000,    "comparison": 2000}
}

In [7]:
# Load Dataset for questions
hotpot_ds = load_dataset("hotpotqa/hotpot_qa", "distractor")

# Train questions: only the fields needed for slicing and the invalid-fact check
train_q = hotpot_ds["train"].select_columns(["id", "type", "level", "supporting_facts"]).to_pandas()

# Dev: IDs only, used for a single leakage check (no Dev labels in this notebook)
validation_ids = set(hotpot_ds["validation"]["id"])

# Corpus: only the columns the invalid-fact check needs
corpus = pd.read_parquet(CORPUS_PATH, columns=["title", "sentence_list"])

# title -> sentence counts of its version(s)
title_to_lens = defaultdict(list)
for title, sentence_list in zip(corpus.title, corpus.sentence_list):
    title_to_lens[title].append(len(sentence_list))

def has_invalid_fact(sf) -> bool:
    return any(
        not any(sent_id < n for n in title_to_lens.get(title, []))
        for title, sent_id in zip(sf["title"], sf["sent_id"])
    )

invalid_ids = set(train_q.id[train_q.supporting_facts.map(has_invalid_fact)])
n_invalid_hard = train_q.id.isin(invalid_ids)[train_q.level == "hard"].sum()

print(f"Train questions with an invalid supporting fact: {len(invalid_ids)} ({n_invalid_hard} hard)")
assert len(invalid_ids) == 22 and n_invalid_hard == 4

Train questions with an invalid supporting fact: 22 (4 hard)


In [8]:
# Pool: hard train questions without invalid supporting facts, in a fixed order
pool = train_q[(train_q.level == "hard") & ~train_q.id.isin(invalid_ids)]
pool = pool[["id", "type"]].sort_values("id").reset_index(drop=True)

# Shuffle each type once, then cut consecutive pieces in SLICE_COUNTS order
shuffled = {
    qtype: group.sample(frac=1, random_state=SEED).id.tolist()
    for qtype, group in pool.groupby("type")
}

slices = {}
start = {qtype: 0 for qtype in shuffled}
for name, counts in SLICE_COUNTS.items():
    ids = []
    for qtype, n in counts.items():
        piece = shuffled[qtype][start[qtype] : start[qtype] + n]
        assert len(piece) == n, f"not enough {qtype} questions for {name}"
        ids += piece
        start[qtype] += n
    slices[name] = ids

# Whatever is left becomes the reserve
slices["reserve"] = [qid for qtype in shuffled for qid in shuffled[qtype][start[qtype]:]]

# Mix types within each slice so a run cut short still has both types
slices = {
    name: pd.Series(ids).sample(frac=1, random_state=SEED).tolist()
    for name, ids in slices.items()
}

# Checks
id_to_type = dict(zip(pool.id, pool.type))
all_ids = [qid for ids in slices.values() for qid in ids]
assert len(all_ids) == len(set(all_ids)), "slices overlap"
assert set(all_ids) == set(pool.id), "slices do not cover the pool exactly"
assert not set(all_ids) & validation_ids, "Dev question in a slice"
for name, counts in SLICE_COUNTS.items():
    got = pd.Series([id_to_type[q] for q in slices[name]]).value_counts().to_dict()
    assert got == counts, f"{name}: {got} != {counts}"

# Save: one JSON list of IDs per slice + a manifest
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
for name, ids in slices.items():
    with open(SPLITS_DIR / f"{name}.json", "w") as f:
        json.dump(ids, f, indent=0)

In [9]:
manifest = {
    "source": "hotpotqa/hotpot_qa distractor, train split",
    "filter": "level == 'hard', excluding questions with invalid supporting facts",
    "excluded_invalid_supporting_facts": int(train_q.id.isin(invalid_ids)[train_q.level == "hard"].sum()),
    "pool_size": len(pool),
    "seed": SEED,
    "counts": {
        name: pd.Series([id_to_type[q] for q in ids]).value_counts().to_dict()
        for name, ids in slices.items()
    },
}
with open(SPLITS_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest["counts"], indent=2))

{
  "pilot": {
    "bridge": 160,
    "comparison": 40
  },
  "tune": {
    "bridge": 800,
    "comparison": 200
  },
  "probe_train": {
    "bridge": 8000,
    "comparison": 2000
  },
  "reserve": {
    "bridge": 3487,
    "comparison": 970
  }
}
